In [2]:
from datasets import load_dataset
ds = load_dataset("ucirvine/sms_spam")

In [3]:
import pandas as pd

df = pd.DataFrame(ds['train']).head(10)
print(df)

                                                 sms  label
0  Go until jurong point, crazy.. Available only ...      0
1                    Ok lar... Joking wif u oni...\n      0
2  Free entry in 2 a wkly comp to win FA Cup fina...      1
3  U dun say so early hor... U c already then say...      0
4  Nah I don't think he goes to usf, he lives aro...      0
5  FreeMsg Hey there darling it's been 3 week's n...      1
6  Even my brother is not like to speak with me. ...      0
7  As per your request 'Melle Melle (Oru Minnamin...      0
8  WINNER!! As a valued network customer you have...      1
9  Had your mobile 11 months or more? U R entitle...      1


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.35,
    random_state=1,
    stratify=df["label"],
)
train_df = train_df.copy()
test_df = test_df.copy()

print(f"Training messages: {len(train_df)}")
print(f"Testing messages:  {len(test_df)}")

Training messages: 6
Testing messages:  4


In [8]:
import re
import numpy as np

label_name = {0: "ham", 1: "spam"}

# creates a compiled regular expression object that matches one or more alphanumeric words
# separated by single spaces, while ignoring underscores and punctuation
token_pattern = re.compile(r"[^\W_]+(?:['’][^\W_]+)*", flags=re.UNICODE) 

def tokenize(text):
    """Return normalized word and number tokens from one message."""
    if pd.isna(text):    # isna() detect the missing value
        return []
    return token_pattern.findall(str(text).casefold())

unknown_labels = set(df["label"].dropna().unique()) - set(label_name)
if unknown_labels:
    raise ValueError(f"Unexpected dataset labels: {unknown_labels}")

train_df["tokens"] = train_df["sms"].map(tokenize) # map() -> Take each indivisual entry in column and pass through tokenize function
train_df[["sms", "tokens", "label"]]

,sms,tokens,label
6,Even my brother is not like to speak with me. ...,"[even, my, brother, is, not, like, to, speak, ...",0
0,"Go until jurong point, crazy.. Available only ...","[go, until, jurong, point, crazy, available, o...",0
2,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f...",1
1,Ok lar... Joking wif u oni...\n,"[ok, lar, joking, wif, u, oni]",0
8,WINNER!! As a valued network customer you have...,"[winner, as, a, valued, network, customer, you...",1
3,U dun say so early hor... U c already then say...,"[u, dun, say, so, early, hor, u, c, already, t...",0


In [11]:
token_rows = train_df[["label", "tokens"]].explode("tokens").dropna(subset=["tokens"])
word_counts = token_rows.groupby(["label", "tokens"], observed=True).size().rename("count").reset_index()
word_counts.assign(class_name=word_counts["label"].map(label_name))[
    ["class_name", "tokens", "count"]
]

,class_name,tokens,count
0,ham,aids,1
1,ham,already,1
2,ham,amore,1
3,ham,available,1
4,ham,brother,1
...,...,...,...
92,spam,valued,1
93,spam,win,1
94,spam,winner,1
95,spam,wkly,1


In [12]:
class_counts = train_df["label"].value_counts().sort_index()   # index: 0 = ham, 1 = spam
priors = class_counts / class_counts.sum()

p_ham = priors[0]
p_spam = priors[1]
print(f"P(ham)  = {p_ham:.4f}")
print(f"P(spam) = {p_spam:.4f}")

P(ham)  = 0.6667
P(spam) = 0.3333


In [13]:
# Find Word probabilities on spam and ham massege, i.e P(word | spam) & P(word | ham)
def word_probability(word_counts, k=0.5):
  
  count_table = (word_counts.pivot(index="tokens", columns="label", values="count").fillna(0))
  count_table.columns = ["ham_count", "spam_count"]       # 0 -> ham, 1 -> spam

  vocab_size = len(count_table)                            # V = number of distinct words

  total_ham_words = count_table["ham_count"].sum()
  total_spam_words = count_table["spam_count"].sum()

# P(word | class) = (count(word, class) + alpha) / (total words in class + alpha * V)
  count_table["P(word|ham)"] = (count_table["ham_count"] + k) / (total_ham_words + k * vocab_size)
  count_table["P(word|spam)"] = (count_table["spam_count"] + k) / (total_spam_words + k * vocab_size)

  return count_table.sort_values("spam_count", ascending=False)

print(word_probability(word_counts, k=0.5))

        ham_count  spam_count  P(word|ham)  P(word|spam)
tokens                                                  
to            1.0         5.0        0.015      0.052885
a             0.0         2.0        0.005      0.024038
entry         0.0         2.0        0.005      0.024038
fa            0.0         2.0        0.005      0.024038
claim         0.0         2.0        0.005      0.024038
...           ...         ...          ...           ...
until         1.0         0.0        0.015      0.004808
wat           1.0         0.0        0.015      0.004808
wif           1.0         0.0        0.015      0.004808
with          1.0         0.0        0.015      0.004808
world         1.0         0.0        0.015      0.004808

[94 rows x 4 columns]


In [18]:
# Find P(spam | word) and P (ham | word)
word_probs = word_probability(word_counts, k=0.5)

def spam_probability(test_df, k=0.5):
  message_word = tokenize(test_df)

  if not message_word:
        return pd.DataFrame(columns=["in_vocab", "P(word|spam)", "P(word|ham)",
                                     "P(spam|word)", "P(ham|word)"])
  
  word_rows = word_probs.reindex(message_word).copy()
  word_rows["in_vocab"] = word_rows["P(word|spam)"].notna()

  vocab_size = len(word_probs)
  total_ham_words = word_probs["ham_count"].sum()
  total_spam_words = word_probs["spam_count"].sum()

  word_rows["P(word|spam)"] = word_rows["P(word|spam)"].fillna(
      k / (total_spam_words + k * vocab_size)
  )
  word_rows["P(word|ham)"] = word_rows["P(word|ham)"].fillna(
      k / (total_ham_words + k * vocab_size)
  )
  
  spam_weight = word_rows["P(word|spam)"] * p_spam
  ham_weight = word_rows["P(word|ham)"] * p_ham
  denominator = spam_weight + ham_weight

  word_rows["P(spam|word)"] = spam_weight / denominator
  word_rows["P(ham|word)"] = ham_weight / denominator
  return word_rows[["in_vocab", "P(word|spam)", "P(word|ham)",
                    "P(spam|word)", "P(ham|word)"]]
  
for idx, row in test_df.iterrows():
    print(f"Message {idx} (true label: {label_name[row['label']]}):")
    print(spam_probability(row["sms"]))
print(f"Probability of new massage: \n{spam_probability("free zorblax call now")}")

Message 5 (true label: spam):
         in_vocab  P(word|spam)  P(word|ham)  P(spam|word)  P(ham|word)
tokens                                                                 
freemsg     False      0.004808        0.005      0.324675     0.675325
hey         False      0.004808        0.005      0.324675     0.675325
there        True      0.004808        0.015      0.138122     0.861878
darling     False      0.004808        0.005      0.324675     0.675325
it's        False      0.004808        0.005      0.324675     0.675325
been         True      0.014423        0.005      0.590551     0.409449
3           False      0.004808        0.005      0.324675     0.675325
week's      False      0.004808        0.005      0.324675     0.675325
now         False      0.004808        0.005      0.324675     0.675325
and         False      0.004808        0.005      0.324675     0.675325
no          False      0.004808        0.005      0.324675     0.675325
word        False      0.004808   

In [20]:
def classify_message(test_df):
    rows = spam_probability(test_df, k=0.5)
    if rows.empty:
        return {"P(spam)": p_spam, "P(ham)": p_ham}

    log_spam = np.log(p_spam) + np.log(rows["P(word|spam)"]).sum()
    log_ham  = np.log(p_ham)  + np.log(rows["P(word|ham)"]).sum()

    # normalize in a numerically stable way
    m = max(log_spam, log_ham)
    p_s = np.exp(log_spam - m) / (np.exp(log_spam - m) + np.exp(log_ham - m))
    return {"P(spam)": p_s, "P(ham)": 1 - p_s}

for idx, row in test_df.iterrows():
    probabilities = classify_message(row["sms"])
    predicted_label = int(probabilities["P(spam)"] >= 0.5)

    print(
        f"Message {idx}: predicted {label_name[predicted_label]}, "
        f"actual {label_name[row['label']]}, "
        f"P(spam)={probabilities['P(spam)']:.4f}"
    )

Message 5: predicted spam, actual spam, P(spam)=0.5251
Message 9: predicted spam, actual spam, P(spam)=0.7348
Message 4: predicted spam, actual ham, P(spam)=0.5240
Message 7: predicted spam, actual ham, P(spam)=0.9470
